## Ugradnje (Embeddings)

U našem prethodnom primjeru radili smo s visokodimenzionalnim vektorima vrste bag-of-words dužine `vocab_size`, i izričito smo pretvarali niskodimenzionalne vektore pozicijskog predstavljanja u rijetku one-hot reprezentaciju. Ova one-hot reprezentacija nije memorijski učinkovita, a osim toga, svaka se riječ tretira neovisno jedna od druge, tj. one-hot kodirani vektori ne izražavaju nikakvu semantičku sličnost između riječi.

U ovom dijelu nastavit ćemo istraživati **News AG** skup podataka. Za početak, učitajmo podatke i dohvatimo neke definicije iz prethodnog bilježnika.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Što je embedding?

Ideja **embeddinga** je predstaviti riječi pomoću gusto raspoređenih vektora niže dimenzionalnosti, koji na neki način odražavaju semantičko značenje riječi. Kasnije ćemo raspraviti kako izgraditi smisleni word embedding, ali za sada razmislimo o embeddingu kao o načinu smanjenja dimenzionalnosti vektora riječi.

Dakle, embedding sloj će uzeti riječ kao ulaz i proizvesti izlazni vektor određene veličine `embedding_size`. U nekom smislu, vrlo je sličan sloju `Linear`, ali umjesto da uzima one-hot kodirani vektor, moći će uzeti broj riječi kao ulaz.

Korištenjem embedding sloja kao prvog sloja u našoj mreži, možemo preći sa bag-of-words modela na model **embedding bag**, gdje prvo pretvaramo svaku riječ u tekstu u odgovarajući embedding, a zatim računamo neku agregatnu funkciju nad svim tim embeddingima, poput `sum`, `average` ili `max`.

![Slika koja prikazuje embedding klasifikator za pet riječi u sekvenci.](../../../../../translated_images/hr/embedding-classifier-example.b77f021a7ee67eee.webp)

Naša neuronska mreža klasifikatora će početi s embedding slojem, zatim slojem za agregaciju, i linearnim klasifikatorom na vrhu:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Rukovanje varijabilnom veličinom sekvenci

Kao rezultat ove arhitekture, minibatch-evi za našu mrežu morali bi biti kreirani na određeni način. U prethodnoj jedinici, prilikom korištenja bag-of-words, svi BoW tenzori u minibatchu imali su jednaku veličinu `vocab_size`, bez obzira na stvarnu duljinu naše tekstualne sekvence. Kad prijeđemo na word embeddings, završili bismo s varijabilnim brojem riječi u svakom tekstualnom uzorku, i prilikom spajanja tih uzoraka u minibatcheve morali bismo primijeniti padding.

To se može učiniti koristeći istu tehniku pružanja `collate_fn` funkcije data source-u:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Trening klasifikatora ugradnje

Sada kada smo definirali odgovarajući dataloader, možemo trenirati model koristeći funkciju treninga koju smo definirali u prethodnoj jedinici:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Napomena**: Ovdje treniramo samo za 25 tisuća zapisa (manje od jedne pune epohe) zbog uštede vremena, ali možete nastaviti trenirati, napisati funkciju za treniranje kroz nekoliko epoha i eksperimentirati s parametrima brzine učenja kako biste postigli veću točnost. Trebali biste moći doseći točnost od oko 90%.


### EmbeddingBag sloj i prikaz sekvenci varijabilne duljine

U prethodnoj arhitekturi morali smo popunjavati sve sekvence do iste duljine kako bismo ih mogli smjestiti u minibatch. Ovo nije najučinkovitiji način predstavljanja sekvenci varijabilne duljine - drugi pristup bio bi korištenje **offset** vektora, koji bi držao offsete svih sekvenci pohranjenih u jednom velikom vektoru.

![Slika koja prikazuje prikaz sekvence s offsetom](../../../../../translated_images/hr/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Napomena**: Na slici iznad prikazujemo sekvencu znakova, ali u našem primjeru radimo sa sekvencama riječi. Međutim, opći princip predstavljanja sekvenci s offset vektorom ostaje isti.

Za rad s offset predstavom koristimo [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) sloj. Sličan je sloju `Embedding`, ali uzima sadržajni vektor i offset vektor kao ulaz, a također uključuje sloj za prosjekovanje, koji može biti `mean`, `sum` ili `max`.

Evo modificirane mreže koja koristi `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Za pripremu skupa podataka za treniranje, potrebno je pružiti funkciju konverzije koja će pripremiti vektor pomaka:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Imajte na umu da, za razliku od svih prethodnih primjera, naša mreža sada prihvaća dva parametra: vektor podataka i vektor pomaka, koji su različitih veličina. Slično tome, naš učitavač podataka također nam daje 3 vrijednosti umjesto 2: i tekstualni i vektori pomaka se daju kao značajke. Stoga moramo malo prilagoditi našu funkciju treniranja da se time pozabavi:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantički ugrađeni prikazi: Word2Vec

U našem prethodnom primjeru, sloj za ugradnju modela naučio je mapirati riječi u vektorsku reprezentaciju, no ta reprezentacija nije imala mnogo semantičkog značenja. Bilo bi korisno naučiti takvu vektorsku reprezentaciju da slične riječi ili sinonimi odgovaraju vektorima koji su bliski jedni drugima u smislu neke vektorske udaljenosti (npr. euklidske udaljenosti).

Za to, trebamo unaprijed trenirati naš model ugradnje na velikoj zbirci teksta na specifičan način. Jedan od prvih načina za treniranje semantičkih ugradnji zove se [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Temelji se na dvije glavne arhitekture koje se koriste za stvaranje distribuirane reprezentacije riječi:

 - **Continuous bag-of-words** (CBoW) — u ovoj arhitekturi treniramo model da predvidi riječ iz okolnog konteksta. Za dati ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, cilj modela je predvidjeti $W_0$ iz $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** je suprotan CBoW-u. Model koristi okolni prozor kontekstualnih riječi za predviđanje trenutačne riječi.

CBoW je brži, dok je skip-gram sporiji, ali bolje predstavlja rjeđe riječi.

![Slika koja prikazuje oba algoritma, CBoW i Skip-Gram, za pretvaranje riječi u vektore.](../../../../../translated_images/hr/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Za eksperimentiranje s word2vec ugrađivanjem unaprijed treniranim na Google News skupu podataka, možemo koristiti **gensim** biblioteku. Ispod nalazimo riječi najsličnije riječi 'neural'

> **Napomena:** Kad prvi put stvarate vektore riječi, njihovo preuzimanje može potrajati!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Također možemo računati ugrađivanja vektora iz riječi, koja se koriste u treniranju modela klasifikacije (za jasnoću prikazujemo samo prvih 20 komponenti vektora):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Sjajna stvar kod semantičkih ugrađivanja je što možete manipulirati vektorskim kodiranjem kako biste promijenili semantiku. Na primjer, možemo tražiti riječ čija bi vektorska reprezentacija bila što bliža riječima *king* i *woman*, a što udaljenija od riječi *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

I CBoW i Skip-Grams su "prediktivna" ugrađivanja, u tome što uzimaju u obzir samo lokalne kontekste. Word2Vec ne koristi prednosti globalnog konteksta. 

**FastText** se nadograđuje na Word2Vec učenjem vektorskih reprezentacija za svaku riječ i n-grama znakova unutar svake riječi. Vrijednosti reprezentacija se zatim u prosjeku spajaju u jedan vektor pri svakom koraku treniranja. Iako ovo dodaje mnogo dodatnog računa prilikom pred-treniranja, omogućuje ugrađivanje riječi da kodira informacije unutar riječi. 

Druga metoda, **GloVe**, koristi ideju matrice pojavljivanja istovremeno, koristi neuronske metode za razlaganje matrice pojavljivanja u izražajnije i nelinearne vektore riječi.

Možete isprobati primjer mijenjajući ugrađivanja za FastText i GloVe, budući da gensim podržava nekoliko različitih modela ugradnje riječi.


## Korištenje prethodno obučenih ugniježdenja u PyTorchu

Možemo prilagoditi gornji primjer kako bismo unaprijed popunili matricu u našem sloju ugniježdenja semantičkim ugniježdenjima, poput Word2Veca. Moramo uzeti u obzir da se vokabulari prethodno obučenog ugniježdenja i našeg tekstualnog korpusa vjerojatno neće podudarati, pa ćemo težine za nedostajuće riječi inicijalizirati slučajnim vrijednostima:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Sada trenirajmo naš model. Imajte na umu da je vrijeme potrebno za treniranje modela znatno duže nego u prethodnom primjeru, zbog veće veličine sloja za ugrađivanje, a samim time i znatno većeg broja parametara. Također, zbog toga možda ćemo morati trenirati naš model na većem broju primjera ako želimo izbjeći prenaučenost.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

U našem slučaju ne vidimo veliki porast točnosti, što je vjerojatno zbog prilično različitih vokabulara. 
Kako bismo prevladali problem različitih vokabulara, možemo koristiti jedno od sljedećih rješenja:
* Ponovno trenirati word2vec model na našem vokabularu
* Učitati naš skup podataka s vokabularom iz prethodno istreniranog word2vec modela. Vokabular koji se koristi za učitavanje skupa podataka može se specificirati tijekom učitavanja.

Potonji pristup čini se lakšim, osobito zato što PyTorch `torchtext` okvir sadrži ugrađenu podršku za ugrađivanja. Na primjer, možemo instancirati vokabular temeljen na GloVe-u na sljedeći način:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Učitani rječnik ima sljedeće osnovne operacije:
* `vocab.stoi` rječnik nam omogućuje pretvaranje riječi u njezin indeks u rječniku
* `vocab.itos` radi suprotno - pretvara broj u riječ
* `vocab.vectors` je niz vektora ugradnje, tako da za dobivanje ugradnje riječi `s` trebamo koristiti `vocab.vectors[vocab.stoi[s]]`

Evo primjera manipulacije ugrađenim vektorima za demonstraciju jednadžbe **dobar-čovjek+žena = kraljica** (morao sam malo prilagoditi koeficijent da bi funkcioniralo):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Da bismo trenirali klasifikator koristeći te ugrađene podatke, prvo moramo kodirati naš skup podataka koristeći GloVe vokabular:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Kao što smo vidjeli gore, svi vektorski ugradbeni elementi pohranjeni su u `vocab.vectors` matrici. To čini vrlo jednostavnim učitavanje tih težina u težine sloja za ugradnju koristeći jednostavno kopiranje:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Sada naučimo naš model i vidimo hoćemo li dobiti bolje rezultate:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Jedan od razloga zbog kojih ne vidimo značajno povećanje točnosti je taj što neke riječi iz našeg skupa podataka nedostaju u unaprijed istreniranoj GloVe vokabularu, te su stoga u biti ignorirane. Kako bismo prevladali ovaj problem, možemo istrenirati vlastite ugrađene reprezentacije na našem skupu podataka. 


## Kontekstualni ugradbeni prikazi

Jedno ključno ograničenje tradicionalnih unaprijed treniranih ugradbenih prikaza poput Word2Veca je problem razlučivanja značenja riječi. Iako unaprijed trenirani ugradbeni prikazi mogu uhvatiti dio značenja riječi u kontekstu, svako moguće značenje riječi kodirano je u isti ugradbeni prikaz. To može izazvati probleme u modelima koji dolaze nakon toga, jer mnoge riječi poput riječi 'play' imaju različita značenja ovisno o kontekstu u kojem se koriste.

Na primjer, riječ 'play' u ove dvije različite rečenice ima sasvim različito značenje:
- Išao sam na **igru** u kazalištu.
- John želi **igrati** se s prijateljima.

Gore navedeni unaprijed trenirani ugradbeni prikazi predstavljaju oba ova značenja riječi 'play' u istom ugradbenom prikazu. Kako bismo prevladali ovo ograničenje, trebamo graditi ugradbene prikaze temeljene na **jezičnom modelu**, koji je treniran na velikom korpusu teksta i *zna* kako se riječi mogu složiti u različitim kontekstima. Rasprava o kontekstualnim ugradbenim prikazima izvan je opsega ovog vodiča, ali ćemo se vratiti na njih kada budemo govorili o jezičnim modelima u sljedećem poglavlju.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
